# Exploratory analyses of existing algorithms on the KickCap dataset
*Set Kernel as the kickcap environment you installed thanks to the `environment.yml` file at the project root.*

From this notebook, you can preprocess the data.

Then, this notebook runs the command lines needed to train the P2P-Insole algorithm from Watanabe et al. (2025), to predict with the outputed model and to visualize the prediction compared to ground-truth data.

The final part of the notebook is dedicated to a custom code replicating the SolePoser pipeline (Wu et al., 2024), based on a two-stream cross-attention Transformer architecture. 

*(why did we do this: because the simple Transformer from Watanabe et al. (2025) was not able to learn a dynamical mapping between feet pressure and IMU information and whole-body kinematics information. Results: the comparison and, before that, all the tests on the P2P-Insole architecture seem to indicate that the algorithm should be aimed at capturing fine and complex links to be able to learn this mapping in the KickCap dataset. It essentially means that KickCap shows very complex kinematics chains and dynamics, that can only be correctly infered through subtile algorithms.)*

The data provided represent <1% of the full KickCap dataset. Apart from enabling the repository to be submitted to Moodle, this allows fast (but bad) training to allow the user to experience the full pipeline in a relatively short time and with lower computing power.

**Important notes**
1. The tags for files calling are the ones for the minimal data provided. Change if needed.
2. If you run all notebook, be prepared to wait a certain time *(~3 h, ~1 h/model mode with an AMD Ryzen 7 8840HS w/ Radeon 780M Graphics processor)* if you use the minimalist data and if your computer does not have a dedicated graphics card.

In [1]:
import os
from pathlib import Path
import subprocess
import sys

In [2]:
os.chdir(Path.cwd())
os.getcwd()

ROOT = Path.cwd()

# Detect Python executable for cross-platform compatibility
PYTHON_CMD = sys.executable
print(f"Using Python: {PYTHON_CMD}")
print(f"Python version: {sys.version}")

Using Python: c:\Users\joels\anaconda3\python.exe
Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]


## Data pre-processing

In [3]:
PREPROCESS_NB = ROOT / "notebooks" / "usefull_tools" / "data_preprocessing.ipynb"

# Expected files/folders after preprocessing pipeline.
required_paths = [
    ROOT / "data" / "clean_data",
    ROOT / "data" / "clean_data" / "Awinda_targets_soleformer",
    ROOT / "data" / "training_data" / "Insole",
    ROOT / "data" / "training_data" / "skeleton",
    ROOT / "data" / "test_data" / "Insole",
    ROOT / "data" / "test_data" / "skeleton",
]

def _has_runtime_data(root: Path) -> bool:
    train_insole = list((root / "data" / "training_data" / "Insole").glob("Soles_*.txt"))
    train_skel = list((root / "data" / "training_data" / "skeleton").glob("Awinda_*.csv"))
    test_insole = list((root / "data" / "test_data" / "Insole").glob("Soles_*.txt"))
    test_skel = list((root / "data" / "test_data" / "skeleton").glob("Awinda_*.csv"))
    return bool(train_insole and train_skel and test_insole and test_skel)

def run_preprocessing_notebook(preprocess_nb: Path):
    if not preprocess_nb.is_file():
        raise FileNotFoundError(f"Preprocessing notebook not found: {preprocess_nb}")

    cmd = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        "--inplace",
        str(preprocess_nb),
    ]
    print("Running preprocessing notebook:")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True, cwd=str(ROOT))

# Ensure required folders exist before execution check.
for path in required_paths:
    path.mkdir(parents=True, exist_ok=True)

if _has_runtime_data(ROOT):
    print("Preprocessing outputs already detected in runtime folders.")
    print("Skipping re-run. Delete/refresh runtime files if you want a fresh preprocessing pass.")
else:
    run_preprocessing_notebook(PREPROCESS_NB)
    if not _has_runtime_data(ROOT):
        raise RuntimeError(
            "Preprocessing notebook finished but runtime train/test files are still missing. "
            "Open notebooks/usefull_tools/data_preprocessing.ipynb and inspect the last routing/validation cell."
        )
    print("Preprocessing completed and runtime data is ready.")

Preprocessing outputs already detected in runtime folders.
Skipping re-run. Delete/refresh runtime files if you want a fresh preprocessing pass.


From then, run either the `Original P2P-Insole` part, the `Simple sequence-to-sequence` part or the `SoleFormer` part, that are based on three different models to compare their specificities: sequence-to-frame versus sequence-to-sequence inference versus two-stream double-loss cross-attention Transformers. In each part, cells must be run in order, because posterior cells rely on produced files from anterior ones.

## Original P2P-Insole Transformer model

In this mode, additional features can be added (add the wanted instruction in the command lines): 
- time feature: `--use_time_feature <true/false>`
- gradient features: `--use_gradient_data <true/false>`

In [4]:
subprocess.run([PYTHON_CMD, "main.py", "train", "--model_mode", "original"], cwd=str(ROOT), check=True)
print("Model file in results/weight/best_skeleton_model_original.pth")

Model file in results/weight/best_skeleton_model_original.pth


In [5]:
subprocess.run([PYTHON_CMD, "main.py", "predict", "--model_mode", "original", "--checkpoint_file", "results/weight/best_skeleton_model_original.pth"], cwd=str(ROOT), check=True)
print("Prediction file in results/output/Predicted_skeleton_<tag>_original.csv")

Prediction file in results/output/Predicted_skeleton_<tag>_original.csv


In [30]:
subprocess.run([PYTHON_CMD, "main.py", "visual", "--model_mode", "original", "--real_file", "data/test_data/skeleton/Awinda_001CcSs_3_shadow_test.csv", "--pred_file", "results/output/Predicted_skeleton_001CcSs_3_shadow_test_original.csv"], cwd=str(ROOT), check=True)
print("HTML visualization file in results/animation/Animation_<tag>_original.html")

HTML visualization file in results/animation/Animation_<tag>_original.html


## Simple sequence-to-sequence Transformer model
*The original Transformer model Watanabe et al. use is a sequence-to-frame model, that computes only the last frame of the inputed window from all previous frames of the same window. This one directly infer frames of one sequence all at once. The difference lies in the importance we accord to past pressure and feet IMU frames for the prediction of the current frame state.*

In this mode, additional features can be added (add the wanted instruction in the command lines): 
- time feature: `--use_time_feature <true/false>`
- gradient features: `--use_gradient_data <true/false>`

In [7]:
subprocess.run([PYTHON_CMD, "main.py", "train", "--model_mode", "simple_seq2seq"], cwd=str(ROOT), check=True)
print("Model file in results/weight/best_skeleton_model_simple_seq2seq.pth")

Model file in results/weight/best_skeleton_model_simple_seq2seq.pth


In [8]:
subprocess.run([PYTHON_CMD, "main.py", "predict", "--model_mode", "simple_seq2seq", "--checkpoint_file", "results/weight/best_skeleton_model_simple_seq2seq.pth"], cwd=str(ROOT), check=True)
print("Prediction file in results/output/Predicted_skeleton_<tag>_simple_seq2seq.csv")

Prediction file in results/output/Predicted_skeleton_<tag>_simple_seq2seq.csv


In [31]:
subprocess.run([PYTHON_CMD, "main.py", "visual", "--model_mode", "simple_seq2seq", "--real_file", "data/test_data/skeleton/Awinda_001CcSs_3_shadow_test.csv", "--pred_file", "results/output/Predicted_skeleton_001CcSs_3_shadow_test_simple_seq2seq.csv"], cwd=str(ROOT), check=True)
print("HTML visualization file in results/animation/Animation_<tag>_simple_seq2seq.html")

HTML visualization file in results/animation/Animation_<tag>_simple_seq2seq.html


## SoleFormer two-stream double-loss cycle cross-attention Transformer
For this algorithm that produces much better results (not an unmoving skeleton), we provide options for ablation. Meaning, the default is the original SoleFormer pipeline, whereas ablation of certain parts can be overrided in the commands to compare the importance of each algorithm part.

### *Ablation commandlines*

In [10]:
from csv import reader

csv_path = ROOT / "sources" / "ablation_commands.csv"

with csv_path.open(newline="", encoding="utf-8") as f:
    rows = list(reader(f))

col_widths = [max(len(row[i]) for row in rows) for i in range(len(rows[0]))]

for row in rows:
    print(" | ".join(value.ljust(col_widths[i]) for i, value in enumerate(row)))


Category   | Ablation                                    | CommandLine                                                                                                                                                                                                             | Notes                                                                                                                     
Model      | Transformer (IMU only)                      | python main.py train --model_mode original                                                                                                                                                                              | Single-stream baseline from code                                                                                          
Model      | Transformer                                 | python main.py train --model_mode original --use_time_feature false --use_gradient_data false                                                

In [11]:
subprocess.run([PYTHON_CMD, "main.py", "train", "--model_mode", "soleformer"], cwd=str(ROOT), check=True)
print("Model file in results/weight/best_skeleton_model_soleformer.pth")

Model file in results/weight/best_skeleton_model_soleformer.pth


In [12]:
subprocess.run([PYTHON_CMD, "main.py", "predict", "--model_mode", "soleformer", "--checkpoint_file", "results/weight/best_skeleton_model_soleformer.pth"], cwd=str(ROOT), check=True)
print("Prediction file in results/output/Predicted_skeleton_<tag>_soleformer.csv")

Prediction file in results/output/Predicted_skeleton_<tag>_soleformer.csv


In [29]:
subprocess.run([PYTHON_CMD, "main.py", "visual", "--model_mode", "soleformer", "--real_file", "data/test_data/skeleton/Awinda_001CcSs_3_shadow_test.csv", "--pred_file", "results/output/Predicted_skeleton_001CcSs_3_shadow_test_soleformer.csv"], cwd=str(ROOT), check=True)
print("HTML visualization file in results/animation/Animation_<tag>_soleformer.html")

HTML visualization file in results/animation/Animation_<tag>_soleformer.html
